# Validate exports against Uruguay XXI

Checks that the exports in `dua_decoded.parquet` (`TIPO_REGIMEN == "Exportación"`) add up to Uruguay's official goods-export figures, and explains the gap.

## Source to compare against

Uruguay XXI, the government's export-promotion agency, publishes the official series of goods exports, built from DNA's data plus free-zone sources: "Exportaciones de bienes con Zonas Francas" (https://www.uruguayxxi.gub.uy/es/centro-informacion/articulo/exportaciones-de-bienes-con-zonas-francas/). It's an Excel file with one row per month, NCM and destination, from 2001 to the last closed month (the latest month is preliminary). Its `AD/ZF` column tells customs exports (`AD`) from exports out of free zones (`ZF`). The values are FOB US$.

Its own notes (sheet header and monthly reports) say:

- Sources: DNA, Banco Central del Uruguay (electricity exports), Montes del Plata and SiGes Nueva Palmira.
- Since May 2024, "todas las exportaciones se computan en el momento que parten hacia los destinos (sin considerar los envíos hacia las zonas francas)" (Uruguay XXI, Informe mensual de comercio exterior, mayo 2025: https://www.uruguayxxi.gub.uy/uploads/informacion/0b61a6c9e926cf66ff13f8a98193365cc140eef4.pdf). The series is published with this method for every year.

## Why the totals differ, and the adjustment

DUA exports and Uruguay XXI measure different things:

1. **Shipments into Uruguayan free zones are DUA exports**, but not exports for Uruguay XXI. In the DUA they show up with a free zone as destination (`PAIS_PROCEDENCIA_DESTINO` `980`–`997`, except `984`, an oil buoy; names from DNA's `rgpais` table). Most of the value is wood going to the pulp mills.
2. **Exports out of free zones are not in the DUA**: pulp, beverage concentrate, pharmaceuticals and grain shipped from free zones are declared by the free-zone operators, not with a DUA. Uruguay XXI adds them (`AD/ZF == "ZF"`).
3. **Grain through Nueva Palmira** (soy, wheat, rapeseed, barley) is the exception: it reaches the Nueva Palmira free zone with a DUA and leaves from there. Uruguay XXI counts it once, when it leaves; the DUA counts it once, when it enters the free zone. So for these four products the two sides already match (checked below), and no adjustment applies.

Expected Uruguay XXI total = DUA exports − DUA exports into Uruguayan free zones (except grain) + Uruguay XXI free-zone exports (except grain).

The adjustment was derived from the data (the grain exception was found by comparing products), not published by Uruguay XXI as a formula, so the small remaining gap is reported year by year.

## Caching

The Excel file (~26 MB) is downloaded once into `data/raw/uruguayxxi/`, named after its publication date on the server, and never overwritten; a newer publication is saved next to it. Its records are read from the workbook's pivot cache (the visible sheet is only a pivot table) and cached as parquet in `data/interim/uruguayxxi/`.


In [ ]:
import logging
import xml.etree.ElementTree as ET
import zipfile
from datetime import UTC, datetime
from email.utils import parsedate_to_datetime
from pathlib import Path

import polars as pl
import requests

from uruguay_dua_comex import utils
from uruguay_dua_comex.query import DECODED_FILENAME

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(15)

SERIES_URL = (
    "https://www.uruguayxxi.gub.uy/es/centro-informacion/articulo/"
    "exportaciones-de-bienes-con-zonas-francas/?download=es"
)
RAW_DIR = utils.data_raw_dir("uruguayxxi")
INTERIM_DIR = utils.data_interim_dir("uruguayxxi")

# Uruguayan free zones in DNA's country table (rgpais): 980-997, except 984,
# "BOYA PETROLERA JOSE IGNACIO", which is an oil buoy, not a free zone.
UY_FREE_ZONES = [str(code) for code in range(980, 998) if code != 984]

# Grain shipped through the Nueva Palmira free zone (HS4): soy, wheat,
# rapeseed and barley. Counted once on each side, so not adjusted.
GRAIN_HS4 = ["1201", "1001", "1205", "1003"]

# Namespace of the workbook's pivot cache XML.
SPREADSHEET_NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"

In [ ]:
def download_series() -> Path:
    """Download Uruguay XXI's export series, unless that publication is cached.

    The file is named after the server's Last-Modified date, so a newer
    publication is saved next to the old one instead of overwriting it.

    Returns:
        Path to the local Excel file.
    """
    head = requests.head(SERIES_URL, allow_redirects=True, timeout=60)
    head.raise_for_status()
    published = parsedate_to_datetime(head.headers["Last-Modified"])
    path = RAW_DIR / f"exportaciones_con_zf_{published:%Y%m%d}.xlsx"
    if path.exists():
        logger.info("Usando la copia local: %s", path)
        return path

    logger.info("Descargando %s...", SERIES_URL)
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(f"{path.name}.tmp")
    try:
        response = requests.get(SERIES_URL, timeout=300)
        response.raise_for_status()
        tmp_path.write_bytes(response.content)
        tmp_path.replace(path)
    finally:
        tmp_path.unlink(missing_ok=True)
    logger.info("Descargado el %s UTC: %s", datetime.now(UTC).isoformat(), path)
    return path


def read_pivot_cache(xlsx_path: Path) -> pl.DataFrame:
    """Read every record of the workbook's pivot cache, caching it as parquet.

    The visible sheet only shows a pivot table; the full data (one row per
    month, NCM and destination) is stored in the pivot cache.

    Args:
        xlsx_path: Local copy of Uruguay XXI's Excel file.

    Returns:
        One row per record, with year, month, NCM, destination, AD/ZF and FOB.
    """
    cache_path = INTERIM_DIR / f"{xlsx_path.stem}.parquet"
    if cache_path.exists():
        return pl.read_parquet(cache_path)

    with zipfile.ZipFile(xlsx_path) as zf:
        names = zf.namelist()
        definition = next(
            n for n in names if n.startswith("xl/pivotCache/pivotCacheDefinition")
        )
        records = next(
            n for n in names if n.startswith("xl/pivotCache/pivotCacheRecords")
        )
        # Each field lists its distinct values once ("shared items"); records
        # point to them by index (<x v="..."/>) or store the value directly.
        fields = []
        for cache_field in ET.fromstring(zf.read(definition)).find(
            f"{SPREADSHEET_NS}cacheFields"
        ):
            shared = cache_field.find(f"{SPREADSHEET_NS}sharedItems")
            items = [item.get("v") for item in shared] if shared is not None else []
            fields.append((cache_field.get("name"), items))
        keep = {"Año", "Mes", "NCM10", "Destino", "AD/ZF", "FOB US$"}
        columns: dict[str, list[str | None]] = {
            name: [] for name, _ in fields if name in keep
        }
        with zf.open(records) as records_file:
            for _, element in ET.iterparse(records_file):
                if element.tag != f"{SPREADSHEET_NS}r":
                    continue
                for (name, items), value in zip(fields, element, strict=True):
                    if name not in keep:
                        continue
                    raw = value.get("v")
                    if value.tag == f"{SPREADSHEET_NS}x" and raw is not None:
                        raw = items[int(raw)]
                    columns[name].append(raw)
                element.clear()

    df_series = pl.DataFrame(columns).select(
        pl.col("Año").cast(pl.Int32).alias("ANIO"),
        pl.col("Mes").cast(pl.Int32).alias("MES"),
        pl.col("NCM10").alias("NCM"),
        pl.col("Destino").alias("DESTINO"),
        pl.col("AD/ZF").alias("AD_ZF"),
        pl.col("FOB US$").cast(pl.Float64).alias("FOB_USD"),
    )
    INTERIM_DIR.mkdir(parents=True, exist_ok=True)
    df_series.write_parquet(cache_path)
    return df_series


df_xxi = read_pivot_cache(download_series()).with_columns(
    pl.col("NCM").str.slice(0, 4).alias("HS4")
)
df_xxi.group_by("AD_ZF").agg(
    pl.len().alias("filas"),
    pl.col("ANIO").min().alias("desde"),
    pl.col("ANIO").max().alias("hasta"),
)

In [ ]:
# DUA exports, from the year Uruguay XXI and the DUA data overlap.
lf_dua = (
    pl.scan_parquet(utils.data_processed_dir(DECODED_FILENAME))
    .filter(pl.col("TIPO_REGIMEN") == "Exportación")
    .with_columns(
        pl.col("ANIO_PRESENTACION").cast(pl.Int32).alias("ANIO"),
        pl.col("FECHA_DUA").dt.month().cast(pl.Int32).alias("MES"),
        pl.col("NCM").str.slice(0, 4).alias("HS4"),
        pl.col("PAIS_PROCEDENCIA_DESTINO").is_in(UY_FREE_ZONES).alias("A_ZONA_FRANCA"),
    )
)
# Compare only the months both sides cover: Uruguay XXI usually publishes a
# month (preliminary) before it's in the DUA table, or the other way around.
period = pl.col("ANIO") * 100 + pl.col("MES")
last_month = min(
    df_xxi.select(period.max()).item(),
    lf_dua.select(period.max()).collect().item(),
)
logger.info("Último mes comparado: %s", last_month)
df_xxi = df_xxi.filter(period <= last_month)
lf_dua = lf_dua.filter(period <= last_month)

# The free-zone destinations found in the exports, with DNA's names.
(
    lf_dua.filter(pl.col("A_ZONA_FRANCA"))
    .group_by("PAIS_PROCEDENCIA_DESTINO", "PAIS_PROCEDENCIA_DESTINO_NOMBRE")
    .agg((pl.col("VALOR_ADUANA").sum() / 1e6).round(0).alias("VALOR_MILLONES_USD"))
    .sort("VALOR_MILLONES_USD", descending=True)
    .collect()
)

## Grain: the two sides already match

For the four grains, total DUA exports (including those into the Nueva Palmira free zone) are compared with Uruguay XXI's total (customs plus free zone). If they match, grain needs no adjustment.

In [ ]:
df_dua_hs4 = (
    lf_dua.group_by("ANIO", "HS4")
    .agg(
        pl.col("VALOR_ADUANA").sum().alias("dua"),
        pl.col("VALOR_ADUANA").filter(pl.col("A_ZONA_FRANCA")).sum().alias("dua_a_zf"),
    )
    .collect()
)
df_xxi_hs4 = df_xxi.group_by("ANIO", "HS4").agg(
    pl.col("FOB_USD").filter(pl.col("AD_ZF") == "AD").sum().alias("xxi_ad"),
    pl.col("FOB_USD").filter(pl.col("AD_ZF") == "ZF").sum().alias("xxi_zf"),
)
df_hs4 = df_dua_hs4.join(df_xxi_hs4, on=["ANIO", "HS4"], how="inner")
is_grain = pl.col("HS4").is_in(GRAIN_HS4)


def millions(column: str) -> pl.Expr:
    """Sum a US$ column and express it in millions, rounded.

    Args:
        column: Column to sum.

    Returns:
        The rounded sum, in US$ millions.
    """
    return (pl.col(column).sum() / 1e6).round(0)


(
    df_hs4.filter(is_grain)
    .with_columns((pl.col("xxi_ad") + pl.col("xxi_zf")).alias("xxi_total"))
    .group_by("ANIO")
    .agg(millions("dua").alias("dua_granos"), millions("xxi_total").alias("xxi_granos"))
    .with_columns(
        ((pl.col("dua_granos") / pl.col("xxi_granos") - 1) * 100)
        .round(1)
        .alias("dif_pct")
    )
    .sort("ANIO")
)

## Yearly reconciliation

All amounts in US$ millions. `dif_pct` is the gap left after the adjustment, relative to Uruguay XXI's total. The last year is partial: both sides are cut at the last month they both cover, and Uruguay XXI's latest month is preliminary.

In [ ]:
df_yearly = (
    df_dua_hs4.join(df_xxi_hs4, on=["ANIO", "HS4"], how="full", coalesce=True)
    .fill_null(0)
    .group_by("ANIO")
    .agg(
        millions("dua").alias("dua_exportaciones"),
        (pl.col("dua_a_zf").filter(~is_grain).sum() / 1e6)
        .round(0)
        .alias("menos_hacia_zf"),
        (pl.col("xxi_zf").filter(~is_grain).sum() / 1e6).round(0).alias("mas_desde_zf"),
        ((pl.col("xxi_ad") + pl.col("xxi_zf")).sum() / 1e6)
        .round(0)
        .alias("uruguay_xxi"),
    )
    .with_columns(
        (
            pl.col("dua_exportaciones")
            - pl.col("menos_hacia_zf")
            + pl.col("mas_desde_zf")
        ).alias("esperado")
    )
    .with_columns(
        (pl.col("esperado") - pl.col("uruguay_xxi")).alias("dif"),
        ((pl.col("esperado") / pl.col("uruguay_xxi") - 1) * 100)
        .round(2)
        .alias("dif_pct"),
    )
    .filter(pl.col("ANIO") >= 2016, pl.col("ANIO") <= last_month // 100)
    .sort("ANIO")
)
df_yearly

## Line-level check

Customs exports only (`AD` on Uruguay XXI's side, every destination but the Uruguayan free zones on the DUA side), compared by year, month and 10-digit NCM. The share of cells within 1% shows whether `VALOR_ADUANA` is the same FOB value Uruguay XXI uses.

In [ ]:
df_cells = (
    lf_dua.filter(~pl.col("A_ZONA_FRANCA"))
    .group_by("ANIO", "MES", "NCM")
    .agg(pl.col("VALOR_ADUANA").sum().alias("dua"))
    .collect()
    .join(
        df_xxi.filter(pl.col("AD_ZF") == "AD")
        .group_by("ANIO", "MES", "NCM")
        .agg(pl.col("FOB_USD").sum().alias("xxi")),
        on=["ANIO", "MES", "NCM"],
        how="inner",
    )
)
df_cells.select(
    pl.len().alias("celdas_en_ambos"),
    ((pl.col("dua") / pl.col("xxi") - 1).abs() < 0.01)
    .mean()
    .round(4)
    .alias("share_dentro_1pct"),
)

## Where the remaining gap is

Products (HS4, outside grain) with the largest customs-side difference over the whole period: DUA exports outside the free zones minus Uruguay XXI's `AD` exports.

In [ ]:
(
    df_hs4.filter(~is_grain, pl.col("ANIO") >= 2016)
    .group_by("HS4")
    .agg(
        ((pl.col("dua") - pl.col("dua_a_zf") - pl.col("xxi_ad")).sum() / 1e6)
        .round(0)
        .alias("dif")
    )
    .sort(pl.col("dif").abs(), descending=True)
    .head(10)
)